# Iterative Verification and Feedback-Driven Repair of LLM-Generated Counterfactual Explanations
XAI project (Track 2) based on *Can LLMs Explain Themselves Counterfactually?* (EMNLP 2025).

**Setup:** SST-2 dataset, a fixed DistilBERT sentiment classifier, Qwen2.5-3B-Instruct as the generator, LangGraph for the repair loop.

**Methods compared:**
1. One-shot generation (baseline)
2. Multi-candidate selection (baseline)
3. Iterative verification and repair (proposed)
4. Iterative repair with feedback history: the LLM sees all of its earlier failed attempts for the sentence

**Metrics:** validity, edit distance, LLM calls and token usage, plus semantic similarity, fluency, an LLM-judge plausibility score and an actionability rate.

Run with a GPU: Runtime > Change runtime type > T4 GPU.

## Setup : Install dependencies

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes datasets langgraph pygraphviz sentence-transformers

## Imports & Experiment settings
All the settings are in one place so they are easy to change.
- `N_EVAL`: held-out examples used to compare all methods
- `K`: candidates per example for multi-candidate selection
- `MAX_ATTEMPTS`: retry limit for iterative repair (equal to `K`, so the maximum budgets match)
- `SIM_MIN`, `REL_EDIT_MAX`, `PPL_RATIO_MAX`, `JUDGE_MIN`, `LEN_RATIO_MIN`, `LEN_RATIO_MAX`: thresholds that decide whether a valid counterfactual is also actionable
- `ROBUST_CONF`: classifier confidence needed to call a flip robust

In [ ]:
import re, time, random, operator
from typing import TypedDict, Annotated, List, Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
N_EVAL = 250
K = 5
MAX_ATTEMPTS = 5
MAX_NEW_TOKENS = 100
SIM_MIN = 0.6
REL_EDIT_MAX = 0.5
PPL_RATIO_MAX = 3.0
JUDGE_MIN = 3.5
LEN_RATIO_MIN = 0.5
LEN_RATIO_MAX = 1.5
ROBUST_CONF = 0.9

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
print("GPU available:", torch.cuda.is_available())

## Load SST-2 and the fixed classifier
The classifier is the model being explained. It is never changed. We only use it to check whether a generated counterfactual reaches the target label.

In [ ]:
from datasets import load_dataset
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

dataset = load_dataset("nyu-mll/glue", "sst2")

classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

print(dataset)
print(classifier("The movie was engaging and beautifully written."))

## Load the generator LLM
Qwen2.5-3B-Instruct in 4-bit, so it fits on a free Colab T4 GPU. This model writes the counterfactuals.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)
llm.eval()

print("LLM loaded successfully!")

## Helper functions
- `detokenize`: SST-2 text is pre-tokenized (`it 's`, `do n't`). We turn it back into normal text so the LLM sees natural sentences and edit distance is not inflated by spacing.
- `word_edit_distance`: number of word insertions, deletions and substitutions between the original and the counterfactual (punctuation and case ignored).
- `clean_output`: removes quotes or labels the LLM sometimes adds.
- `predict_sentiment`: asks the fixed classifier for its label and confidence. This is the verification step.

In [ ]:
def detokenize(text):
    text = text.strip()
    for a, b in [(" n't", "n't"), (" 's", "'s"), (" 're", "'re"), (" 've", "'ve"),
                 (" 'm", "'m"), (" 'll", "'ll"), (" 'd", "'d")]:
        text = text.replace(a, b)
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

def words(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower())

def word_edit_distance(original, candidate):
    a, b = words(original), words(candidate)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[-1][-1]

def clean_output(text):
    text = text.strip().split("\n")[0].strip()
    text = re.sub(r"^(rewritten|revised) (review|sentence)?\s*:\s*", "", text, flags=re.I)
    if len(text) > 1 and text[0] in "\"'" and text[-1] in "\"'":
        text = text[1:-1].strip()
    return text

def predict_sentiment(text):
    if not text.strip():
        return "NONE", 0.0
    result = classifier(text, truncation=True)[0]
    return result["label"].upper(), result["score"]

print("Testing Functions ")
print("-------------------")
print(detokenize("it 's a charming and often affecting journey ."))
print(word_edit_distance("The movie was slow.", "The movie was great and fast."))
print(predict_sentiment("The movie was slow and predictable."))

## Asking the LLM
`ask_llm` sends one prompt to Qwen and returns the answer plus what it cost:
- the cleaned answer text
- number of prompt tokens
- number of new tokens generated
- seconds taken

`sample=False` means greedy decoding (same answer every time). `sample=True` means random sampling, which gives a different answer each call. We need this for multi-candidate selection and for repair attempts.

In [ ]:
def ask_llm(prompt, sample=False):
    messages = [
        {"role": "system", "content": "You rewrite movie review sentences."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    n_prompt = inputs["input_ids"].shape[1]

    start = time.time()
    with torch.no_grad():
        if sample:
            output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                  temperature=0.9, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
        else:
            output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    seconds = time.time() - start

    new_ids = output[0][n_prompt:]
    answer = clean_output(tokenizer.decode(new_ids, skip_special_tokens=True))
    return answer, n_prompt, len(new_ids), seconds

## Prepare the data
- We sample `N_EVAL` examples from the SST-2 **validation** split with a fixed seed.
- The **target label** is always the opposite of what the classifier predicts for the original sentence. Every method uses the same rule and the same examples.

In [ ]:
val = dataset["validation"].to_pandas()
eval_df = val.sample(n=N_EVAL, random_state=SEED).reset_index(drop=True)

eval_df["id"] = eval_df["idx"]
eval_df["original"] = eval_df["sentence"].map(detokenize)

# classifier prediction on each original sentence
preds = classifier(eval_df["original"].tolist(), truncation=True, batch_size=32)
eval_df["original_label"] = [p["label"].upper() for p in preds]
eval_df["target_label"] = np.where(eval_df["original_label"] == "NEGATIVE", "POSITIVE", "NEGATIVE")

print("Evaluation examples:", len(eval_df))
print(eval_df["target_label"].value_counts())
eval_df[["id", "original", "original_label", "target_label"]].head()

## Prompts
All methods use the **same initial prompt**, so differences in results come from the method and not from different wording.

- `initial_prompt`: asks for a rewrite with the target sentiment.
- `repair_prompt`: used only by iterative repair. It shows the LLM its failed candidate and what the classifier predicted, and asks for a revision.
- `guidelines` is optional. It stays empty for all baselines and is filled only in the feedback-memory experiments.

In [ ]:
def guideline_text(guidelines):
    if not guidelines:
        return ""
    return f"\nGuidelines learned from earlier failures (apply them where relevant):\n{guidelines}\n"

def initial_prompt(original, target, guidelines=None):
    return (f"Rewrite the movie review sentence so that its sentiment is {target.lower()}.\n"
            "Preserve the original meaning and change as little text as possible.\n"
            "Return only the rewritten sentence.\n"
            f"{guideline_text(guidelines)}\n"
            f"Original review: {original}\n"
            "Rewritten review:")

def repair_prompt(original, target, previous, observed_label, observed_conf, guidelines=None):
    return (f"You rewrote a movie review sentence so that its sentiment would be {target.lower()}, "
            f"but a fixed sentiment classifier predicted {observed_label} "
            f"(confidence {observed_conf:.3f}) for your rewrite.\n\n"
            f"Original review: {original}\n"
            f"Your previous rewrite: {previous}\n\n"
            f"Revise the rewrite so that the classifier predicts {target}. "
            "Preserve the original meaning and change as little text as possible.\n"
            "Return only the revised sentence.\n"
            f"{guideline_text(guidelines)}\n"
            "Revised review:")

print(initial_prompt("the film is slow and dull.", "POSITIVE"))
print("-" * 50)
print(repair_prompt("the film is slow and dull.", "POSITIVE", "the film is slow but fun.", "NEGATIVE", 0.91))

answer, n_prompt, n_new, secs = ask_llm(initial_prompt("The movie was slow and predictable.", "POSITIVE"))
print(answer, "|", n_prompt, "prompt tokens |", n_new, "new tokens |", round(secs, 2), "s")
print(predict_sentiment(answer))

## Verification
`verify` is the check at the heart of the project. It sends a candidate to the fixed classifier and records:
- the predicted label and confidence
- whether the prediction equals the target label (`valid`)
- how much the text changed (word edit distance, absolute and relative to the original length)
- `target_confidence`: how much probability the classifier gives to the target label. We use it later to pick the best failed candidate when nothing succeeds.

In [ ]:
def verify(original, target, candidate):
    label, conf = predict_sentiment(candidate)

    if label == target:
        target_conf = conf
    elif label == "NONE":
        target_conf = 0.0
    else:
        target_conf = 1 - conf          # binary classifier

    edit = word_edit_distance(original, candidate)

    return {
        "candidate": candidate,
        "predicted_label": label,
        "confidence": conf,
        "target_confidence": target_conf,
        "valid": label == target,
        "edit_distance": edit,
        "rel_edit_distance": edit / max(len(words(original)), 1),
    }

print(verify("The movie was slow and predictable.", "POSITIVE", "The movie was fast and exciting."))

# Baseline 1: One-shot generation (LangGraph)
Each example flows through the graph as one row of data:

`START -> generate -> verify -> save -> END`

- **start**: receives one row (id, original sentence, target label) and sets the counters to zero.
- **generate**: the LLM writes one counterfactual (greedy).
- **verify**: the fixed classifier checks it.
- **save**: appends the result as one line of `one_shot_results.csv`. There is no revision.

All pipelines share one `State` so the later graphs (multi-candidate, iterative, memory) can reuse it.

In [ ]:
import os
from langgraph.graph import StateGraph, START, END

class State(TypedDict, total=False):
    # the input row
    id: int
    original: str
    target_label: str
    # the current candidate and its verification
    candidate: str
    predicted_label: str
    confidence: float
    target_confidence: float
    valid: bool
    edit_distance: int
    rel_edit_distance: float
    # cost counters
    llm_calls: int
    prompt_tokens: int
    new_tokens: int
    seconds: float
    saved: bool

def start_node(state):
    return {"candidate": "", "llm_calls": 0, "prompt_tokens": 0, "new_tokens": 0, "seconds": 0.0}

def generate_node(state):
    prompt = initial_prompt(state["original"], state["target_label"])
    answer, n_prompt, n_new, secs = ask_llm(prompt)
    return {"candidate": answer,
            "llm_calls": state["llm_calls"] + 1,
            "prompt_tokens": state["prompt_tokens"] + n_prompt,
            "new_tokens": state["new_tokens"] + n_new,
            "seconds": state["seconds"] + secs}

def verify_node(state):
    return verify(state["original"], state["target_label"], state["candidate"])

SAVE_COLUMNS = ["id", "original", "target_label", "candidate", "predicted_label", "valid",
                "edit_distance", "rel_edit_distance", "llm_calls", "prompt_tokens", "new_tokens", "seconds"]

def save_node(state):
    row = {c: state[c] for c in SAVE_COLUMNS}
    pd.DataFrame([row]).to_csv("one_shot_results.csv", mode="a", index=False,
                               header=not os.path.exists("one_shot_results.csv"))
    return {"saved": True}

g = StateGraph(State)
g.add_node("start", start_node)
g.add_node("generate", generate_node)
g.add_node("verify", verify_node)
g.add_node("save", save_node)
g.add_edge(START, "start")
g.add_edge("start", "generate")
g.add_edge("generate", "verify")
g.add_edge("verify", "save")
g.add_edge("save", END)
one_shot_graph = g.compile()

# print(one_shot_graph.get_graph().draw_mermaid())

In [ ]:
# run one-shot on every evaluation example
if os.path.exists("one_shot_results.csv"):
    os.remove("one_shot_results.csv")          # start from a clean file

set_seed()
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    one_shot_graph.invoke({"id": int(row["id"]),
                           "original": row["original"],
                           "target_label": row["target_label"]})

res_one = pd.read_csv("one_shot_results.csv")
print(len(res_one), "rows saved")

In [ ]:
# look at some failures
res_one[~res_one["valid"]][["original", "target_label", "candidate", "predicted_label"]].head(8)

In [ ]:
def summarize(df, name):
    ok = df[df["valid"]]
    return pd.DataFrame([{
        "Method": name,
        "Validity (%)": 100 * df["valid"].mean(),
        "Edit distance (valid)": ok["edit_distance"].mean(),
        "Relative edit (valid)": ok["rel_edit_distance"].mean(),
        "LLM calls / example": df["llm_calls"].mean(),
        "Tokens / example": (df["prompt_tokens"] + df["new_tokens"]).mean(),
        "Seconds / example": df["seconds"].mean(),
    }]).round(2)

summarize(res_one, "One-shot")

# Baseline 2: Multi-candidate selection (LangGraph)
Instead of trusting one generation, we sample **K = 5** independent candidates for each sentence, verify all of them, and keep the best one.

`START -> generate_k -> select -> save -> END`

- **generate_k**: a `for` loop calls the LLM K times (random sampling, so each answer is different) and verifies each candidate with the fixed classifier.
- **select**: among the **valid** candidates, keep the one with the **smallest edit distance**. If none is valid, the example counts as a failure (we only keep the closest candidate so we can inspect it).
- **save**: writes the result to `multi_candidate_results.csv`.

Cost note: K separate LLM calls are made, so calls and tokens are about K times those of one-shot. This is the price of this baseline.

In [ ]:
class MultiState(State):
    candidates: list            # every verified candidate for this example
    n_valid_candidates: int

def generate_k_node(state):
    candidates = []
    calls, p_tokens, n_tokens, secs = state["llm_calls"], state["prompt_tokens"], state["new_tokens"], state["seconds"]

    for i in range(K):
        prompt = initial_prompt(state["original"], state["target_label"])
        answer, n_prompt, n_new, s = ask_llm(prompt, sample=True)
        candidates.append(verify(state["original"], state["target_label"], answer))
        calls += 1
        p_tokens += n_prompt
        n_tokens += n_new
        secs += s

    return {"candidates": candidates, "llm_calls": calls,
            "prompt_tokens": p_tokens, "new_tokens": n_tokens, "seconds": secs}

def select_node(state):
    valid = [c for c in state["candidates"] if c["valid"]]
    if valid:
        best = min(valid, key=lambda c: c["edit_distance"])      # smallest change among valid
    else:
        best = max(state["candidates"], key=lambda c: c["target_confidence"])   # closest failure
    return {**best, "n_valid_candidates": len(valid)}

MULTI_COLUMNS = SAVE_COLUMNS + ["n_valid_candidates"]

def save_multi_node(state):
    row = {c: state[c] for c in MULTI_COLUMNS}
    pd.DataFrame([row]).to_csv("multi_candidate_results.csv", mode="a", index=False,
                               header=not os.path.exists("multi_candidate_results.csv"))
    return {"saved": True}

g = StateGraph(MultiState)
g.add_node("start", start_node)
g.add_node("generate_k", generate_k_node)
g.add_node("select", select_node)
g.add_node("save", save_multi_node)
g.add_edge(START, "start")
g.add_edge("start", "generate_k")
g.add_edge("generate_k", "select")
g.add_edge("select", "save")
g.add_edge("save", END)
multi_graph = g.compile()

# print(multi_graph.get_graph().draw_mermaid())

In [ ]:
if os.path.exists("multi_candidate_results.csv"):
    os.remove("multi_candidate_results.csv")

set_seed()
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    multi_graph.invoke({"id": int(row["id"]),
                        "original": row["original"],
                        "target_label": row["target_label"]})

res_multi = pd.read_csv("multi_candidate_results.csv")
print(len(res_multi), "rows saved")

In [ ]:
pd.concat([summarize(res_one, "One-shot"),
           summarize(res_multi, f"Multi-candidate (K={K})")], ignore_index=True)

In [ ]:
# how many of the K candidates were valid per example, and which examples still failed
print(res_multi["n_valid_candidates"].value_counts().sort_index())
res_multi[~res_multi["valid"]][["original", "target_label", "candidate", "predicted_label"]].head(8)

# Proposed method: Iterative verification and repair (LangGraph)
Instead of accepting the first output, we verify it and, if the classifier does not predict the target label, send the failed candidate back to the LLM together with what the classifier said. The LLM revises, we verify again, and so on until success or the retry limit (`MAX_ATTEMPTS`).

```
START -> generate -> verify ─┬─ valid, or attempts = MAX_ATTEMPTS -> save -> END
                             └─ otherwise -> repair -> verify (loop)
```

- **generate**: first attempt, greedy decoding and the same prompt as one-shot. So the first-attempt validity must equal the one-shot validity, and any improvement comes from repair alone.
- **verify**: fixed classifier check. It also counts the attempt.
- **route** (conditional edge): decides whether to stop or repair.
- **repair**: sees only the **previous candidate** and the **classifier's prediction and confidence** for it. It then writes a revision (sampling, so a retry is not identical to the previous attempt).
- **save**: writes the final result to `iterative_results.csv`.

In [ ]:
class IterState(State):
    attempts: int
    first_attempt_valid: bool

def start_iter_node(state):
    return {"candidate": "", "attempts": 0, "llm_calls": 0,
            "prompt_tokens": 0, "new_tokens": 0, "seconds": 0.0}

def generate_iter_node(state):
    prompt = initial_prompt(state["original"], state["target_label"])
    answer, n_prompt, n_new, secs = ask_llm(prompt)              # greedy, same as one-shot
    return {"candidate": answer, "attempts": 1,
            "llm_calls": state["llm_calls"] + 1,
            "prompt_tokens": state["prompt_tokens"] + n_prompt,
            "new_tokens": state["new_tokens"] + n_new,
            "seconds": state["seconds"] + secs}

def verify_iter_node(state):
    result = verify(state["original"], state["target_label"], state["candidate"])
    if state["attempts"] == 1:
        result["first_attempt_valid"] = result["valid"]
    return result

def route_after_verify(state):
    if state["valid"] or state["attempts"] >= MAX_ATTEMPTS:
        return "save"
    return "repair"

def repair_node(state):
    # only the previous output and the classifier's verdict are given to the LLM
    prompt = repair_prompt(state["original"], state["target_label"], state["candidate"],
                           state["predicted_label"], state["confidence"])
    answer, n_prompt, n_new, secs = ask_llm(prompt, sample=True)
    return {"candidate": answer, "attempts": state["attempts"] + 1,
            "llm_calls": state["llm_calls"] + 1,
            "prompt_tokens": state["prompt_tokens"] + n_prompt,
            "new_tokens": state["new_tokens"] + n_new,
            "seconds": state["seconds"] + secs}

ITER_COLUMNS = SAVE_COLUMNS + ["attempts", "first_attempt_valid"]

def save_iter_node(state):
    row = {c: state[c] for c in ITER_COLUMNS}
    pd.DataFrame([row]).to_csv("iterative_results.csv", mode="a", index=False,
                               header=not os.path.exists("iterative_results.csv"))
    return {"saved": True}

g = StateGraph(IterState)
g.add_node("start", start_iter_node)
g.add_node("generate", generate_iter_node)
g.add_node("verify", verify_iter_node)
g.add_node("repair", repair_node)
g.add_node("save", save_iter_node)
g.add_edge(START, "start")
g.add_edge("start", "generate")
g.add_edge("generate", "verify")
g.add_conditional_edges("verify", route_after_verify, {"repair": "repair", "save": "save"})
g.add_edge("repair", "verify")
g.add_edge("save", END)
iterative_graph = g.compile()

# print(iterative_graph.get_graph().draw_mermaid())

In [ ]:
if os.path.exists("iterative_results.csv"):
    os.remove("iterative_results.csv")

set_seed()
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    iterative_graph.invoke({"id": int(row["id"]),
                            "original": row["original"],
                            "target_label": row["target_label"]},
                           config={"recursion_limit": 50})

res_iter = pd.read_csv("iterative_results.csv")
print(len(res_iter), "rows saved")
print("Attempt-1 validity (should equal one-shot):",
      round(100 * res_iter["first_attempt_valid"].mean(), 1), "vs", round(100 * res_one["valid"].mean(), 1))

In [ ]:
summary = pd.concat([summarize(res_one, "One-shot"),
                     summarize(res_multi, f"Multi-candidate (K={K})"),
                     summarize(res_iter, "Iterative repair")], ignore_index=True)
summary

In [ ]:
# at which attempt did iterative repair succeed?
print(res_iter[res_iter["valid"]]["attempts"].value_counts().sort_index().to_string())
print("Never valid within the limit:", int((~res_iter["valid"]).sum()))

# # examples that failed first but were repaired
fixed = res_iter[(~res_iter["first_attempt_valid"]) & res_iter["valid"]]
print(f"\n{len(fixed)} examples were repaired")
fixed[["original", "target_label", "candidate", "attempts"]].head(1)

# Iterative repair with feedback history (LangGraph)
In plain iterative repair, the LLM only sees its **previous** output. Here the state keeps a list called `history` with **every failed attempt** for the current sentence: what was written, and what the classifier predicted.

The repair prompt shows the LLM that whole list, so it can see everything it already tried and failed, and avoid repeating it.

```
START -> generate -> verify ─┬─ valid, or attempts = MAX_ATTEMPTS -> save -> END
                             └─ otherwise -> repair (sees full history) -> verify (loop)
```

The first attempt is the same greedy generation as one-shot. The only difference from plain iterative repair is what the repair step sees, so any difference in results comes from the history.

In [ ]:
class HistoryState(IterState):
    history: list          # all failed attempts so far for this sentence

def start_hist_node(state):
    return {"candidate": "", "attempts": 0, "llm_calls": 0,
            "prompt_tokens": 0, "new_tokens": 0, "seconds": 0.0, "history": []}

def verify_hist_node(state):
    result = verify_iter_node(state)
    if not result["valid"]:
        note = {"candidate": state["candidate"],
                "predicted_label": result["predicted_label"],
                "confidence": result["confidence"]}
        result["history"] = state["history"] + [note]      # remember this failure
    return result

def history_repair_prompt(original, target, history):
    tried = ""
    for i, h in enumerate(history, 1):
        tried += f"Attempt {i}: {h['candidate']}\n   classifier predicted {h['predicted_label']} (confidence {h['confidence']:.3f})\n"
    return (f"You are rewriting a movie review sentence so that its sentiment is {target.lower()}.\n"
            "A fixed sentiment classifier checks every rewrite, and all of your earlier attempts failed.\n\n"
            f"Original review: {original}\n\n"
            f"Your failed attempts so far:\n{tried}\n"
            "Look at everything you already tried and do not repeat it. "
            f"Write a new rewrite that the classifier will predict as {target}. "
            "Preserve the original meaning and change as little text as possible.\n"
            "Return only the new sentence.\n\n"
            "New rewrite:")

def repair_hist_node(state):
    prompt = history_repair_prompt(state["original"], state["target_label"], state["history"])
    answer, n_prompt, n_new, secs = ask_llm(prompt, sample=True)
    return {"candidate": answer, "attempts": state["attempts"] + 1,
            "llm_calls": state["llm_calls"] + 1,
            "prompt_tokens": state["prompt_tokens"] + n_prompt,
            "new_tokens": state["new_tokens"] + n_new,
            "seconds": state["seconds"] + secs}

def save_hist_node(state):
    row = {c: state[c] for c in ITER_COLUMNS}
    pd.DataFrame([row]).to_csv("iterative_history_results.csv", mode="a", index=False,
                               header=not os.path.exists("iterative_history_results.csv"))
    return {"saved": True}

g = StateGraph(HistoryState)
g.add_node("start", start_hist_node)
g.add_node("generate", generate_iter_node)
g.add_node("verify", verify_hist_node)
g.add_node("repair", repair_hist_node)
g.add_node("save", save_hist_node)
g.add_edge(START, "start")
g.add_edge("start", "generate")
g.add_edge("generate", "verify")
g.add_conditional_edges("verify", route_after_verify, {"repair": "repair", "save": "save"})
g.add_edge("repair", "verify")
g.add_edge("save", END)
history_graph = g.compile()

# print(history_graph.get_graph().draw_mermaid())


In [ ]:
if os.path.exists("iterative_history_results.csv"):
    os.remove("iterative_history_results.csv")

set_seed()
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    history_graph.invoke({"id": int(row["id"]),
                          "original": row["original"],
                          "target_label": row["target_label"]},
                         config={"recursion_limit": 50})

res_hist = pd.read_csv("iterative_history_results.csv")
print(len(res_hist), "rows saved")

In [ ]:
summary = pd.concat([summarize(res_one, "One-shot"),
                     summarize(res_multi, f"Multi-candidate (K={K})"),
                     summarize(res_iter, "Iterative repair (previous output only)"),
                     summarize(res_hist, "Iterative repair (full failure history)")], ignore_index=True)
display(summary)

print("Mean attempts needed:", round(res_iter["attempts"].mean(), 2), "(previous only) vs",
      round(res_hist["attempts"].mean(), 2), "(full history)")
print("Never valid within the limit:", int((~res_iter["valid"]).sum()), "vs", int((~res_hist["valid"]).sum()))

In [ ]:
# examples that failed first and were fixed with the full history
fixed = res_hist[(~res_hist["first_attempt_valid"]) & res_hist["valid"]]
print(len(fixed), "examples were repaired")
fixed[["original", "target_label", "candidate", "attempts"]].head(1)

# Comparing all methods
We put the four result tables together and compare them on the same examples:
- **Validity**: percentage of examples where the classifier predicts the target label
- **Edit distance**: how many words were changed (valid counterfactuals only)
- **Cost**: LLM calls, tokens and time per example

In [ ]:
results = {
    "One-shot": res_one,
    f"Multi-candidate (K={K})": res_multi,
    "Iterative (previous output)": res_iter,
    "Iterative (full history)": res_hist,
}

summary = pd.concat([summarize(df, name) for name, df in results.items()], ignore_index=True)
summary.to_csv("summary_all_methods.csv", index=False)
summary

## Plot 1: validity, modification and cost
- **Left**: how often each method finds a valid counterfactual (higher is better)
- **Middle**: how many words were changed in the valid ones (lower is better)
- **Right**: tokens used per example (lower is cheaper)

In [ ]:
labels = ["One-shot", "Multi-\ncandidate", "Iterative\n(prev. output)", "Iterative\n(full history)"]
colors = ["orange", "tab:blue", "tab:red", "tab:green"]

fig, ax = plt.subplots(1, 3, figsize=(16, 6))

ax[0].bar(labels, summary["Validity (%)"], color=colors)
ax[0].set_ylabel("Validity (%)")
ax[0].set_ylim(0, 110)
ax[0].set_title("Validity")
for i, v in enumerate(summary["Validity (%)"]):
    ax[0].text(i, v + 2, f"{v:.0f}%", ha="center")

ax[1].bar(labels, summary["Edit distance (valid)"], color=colors)
ax[1].set_ylabel("Words changed")
ax[1].set_title("Edit distance (valid counterfactuals)")
for i, v in enumerate(summary["Edit distance (valid)"]):
    ax[1].text(i, v + 0.1, f"{v:.1f}", ha="center")

ax[2].bar(labels, summary["Tokens / example"], color=colors)
ax[2].set_ylabel("Tokens")
ax[2].set_title("Cost: tokens per example")
for i, v in enumerate(summary["Tokens / example"]):
    ax[2].text(i, v + 5, f"{v:.0f}", ha="center")

plt.tight_layout()
plt.savefig("comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## Plot 2: effect of the retry limit
How does validity change as we allow more attempts? For iterative repair, an example counts as a success at limit *k* if a valid counterfactual was found within its first *k* attempts. Limit 1 is the same as one-shot. Multi-candidate selection is shown as a single point at its budget of K candidates.

In [ ]:
limits = list(range(1, MAX_ATTEMPTS + 1))

def validity_at_limit(df, k):
    return 100 * ((df["valid"]) & (df["attempts"] <= k)).mean()

plt.figure(figsize=(7, 4.5))
plt.plot(limits, [validity_at_limit(res_iter, k) for k in limits],
         marker="o", color="tab:red", label="Iterative (previous output)")
plt.plot(limits, [validity_at_limit(res_hist, k) for k in limits],
         marker="o", color="tab:green", label="Iterative (full history)")
plt.scatter([K], [100 * res_multi["valid"].mean()], marker="s", s=90,
            color="tab:blue", label=f"Multi-candidate (K={K})", zorder=5)
plt.scatter([1], [100 * res_one["valid"].mean()], marker="^", s=90,
            color="gray", label="One-shot", zorder=5)

plt.xticks(limits)
plt.xlabel("Retry limit / generation budget")
plt.ylabel("Validity (%)")
plt.ylim(0, 105)
plt.title("Validity vs. retry limit")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig("retry_limit.png", dpi=200, bbox_inches="tight")
plt.show()

## Plot 3: how many examples succeed in each round
For each iterative method, this shows how many examples became valid for the first time at each attempt, and how many were still invalid after the last allowed attempt (`MAX_ATTEMPTS`).

In [ ]:
iter_results = {"Iterative (previous output)": res_iter, "Iterative (full history)": res_hist}
attempt_labels = [f"Attempt {i}" for i in range(1, MAX_ATTEMPTS + 1)] + ["Failed"]

panel_data = {}
for name, d in iter_results.items():
    solved = d[d["valid"]]["attempts"].value_counts().reindex(range(1, MAX_ATTEMPTS + 1), fill_value=0)
    panel_data[name] = solved.tolist() + [int((~d["valid"]).sum())]

top = max(max(v) for v in panel_data.values())

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, (name, vals) in zip(axes, panel_data.items()):
    total = sum(vals)
    bars = ax.bar(attempt_labels, vals, color=["steelblue"] * MAX_ATTEMPTS + ["orange"])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + top * 0.01,
                f"{v}\n({100 * v / total:.0f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_title(name)
    ax.set_xlabel("Generation Attempt")
    ax.set_ylim(0, top * 1.2)
    ax.grid(axis="y", alpha=0.25)

axes[0].set_ylabel("Number of Examples")
fig.suptitle("Iterative Verification: Counterfactuals Validated Across Attempts")
plt.tight_layout()
plt.savefig("iterative_success_progression.png", dpi=300, bbox_inches="tight")
plt.show()

# Additional metrics and actionable counterfactuals
Validity only says that the classifier flipped. It does not say that the counterfactual is a sensible text. A rewrite like *"the film is not dull and boring"* might flip the label but be awkward, or drift away from the original sentence. Following the counterfactual and recourse literature (Wachter et al. 2017, Ustun et al. 2019, Mothilal et al. 2020, Karimi et al. 2022), where a counterfactual should be **valid, close to the input, sparse and plausible/feasible**, we add the metrics below.

| Property | Metric | Tool |
|---|---|---|
| Validity | classifier predicts the target label | DistilBERT (already used) |
| Robustness of the flip | target-label confidence of the classifier | DistilBERT |
| Proximity / sparsity | word edit distance, absolute and relative | already used |
| Meaning preserved | cosine similarity of sentence embeddings | `all-MiniLM-L6-v2` |
| Fluency | perplexity of the counterfactual divided by perplexity of the original | GPT-2 (independent of the generator) |
| Plausibility | 1 to 5 rating, expected value over the rating-token probabilities | Qwen2.5-3B as judge |
| Not degenerate | not a copy of the original, length within `LEN_RATIO_MIN` to `LEN_RATIO_MAX` times the original | simple checks |
| Cost efficiency | LLM calls and tokens per valid and per actionable counterfactual | already counted |

**Actionable counterfactual** = valid **and** passes every check: not a copy, similarity >= `SIM_MIN`, relative edit <= `REL_EDIT_MAX`, perplexity ratio <= `PPL_RATIO_MAX`, length ratio inside the allowed range, judge rating >= `JUDGE_MIN`. In words: a rewrite a real reviewer could plausibly have written, which keeps the original meaning and only changes the opinion.

Notes: the thresholds are design choices, so a sensitivity check is included below. The judge is the same model that generated the text and may be lenient on its own style, which is why fluency is measured with a separate model (GPT-2, on lower-cased text so that the lower-cased SST-2 originals are not penalised).

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

dev = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
fluency_tok = GPT2TokenizerFast.from_pretrained("gpt2")
fluency_lm = GPT2LMHeadModel.from_pretrained("gpt2").to(dev).eval()

print("Metric models loaded")

In [ ]:
def batch_target_conf(texts, targets):
    out = [0.0] * len(texts)
    idx = [i for i, t in enumerate(texts) if t.strip()]
    if idx:
        preds = classifier([texts[i] for i in idx], truncation=True, batch_size=32)
        for i, p in zip(idx, preds):
            out[i] = p["score"] if p["label"].upper() == targets[i] else 1 - p["score"]
    return out

ppl_cache = {}

def perplexity(text):
    text = text.lower().strip()
    if not text:
        return float("nan")
    if text in ppl_cache:
        return ppl_cache[text]
    ids = fluency_tok(text, return_tensors="pt", truncation=True, max_length=256).input_ids
    bos = torch.tensor([[fluency_tok.bos_token_id]])
    ids = torch.cat([bos, ids], dim=1).to(dev)
    with torch.no_grad():
        loss = fluency_lm(input_ids=ids, labels=ids).loss
    ppl_cache[text] = float(torch.exp(loss))
    return ppl_cache[text]

DIGIT_IDS = [tokenizer.encode(str(d), add_special_tokens=False)[0] for d in range(1, 6)]

def judge_prompt(original, candidate, target):
    return (f"Original movie review sentence: {original}\n"
            f"Rewritten sentence: {candidate}\n\n"
            f"The rewrite was produced to change the sentiment to {target.lower()}. "
            "Rate how reasonable the rewrite is as a counterfactual on a scale from 1 to 5.\n"
            "A reasonable rewrite reads as natural, fluent English that a real reviewer could plausibly write, "
            "keeps talking about the same film and aspects as the original, changes only what is needed "
            "to flip the opinion, and does not contradict itself.\n"
            "5 = fully natural and realistic, 3 = acceptable but awkward or partly inconsistent, "
            "1 = unnatural, nonsensical or unrelated.\n"
            "Answer with a single digit.")

def judge_score(original, candidate, target):
    if not candidate.strip():
        return 1.0
    messages = [
        {"role": "system", "content": "You are a strict evaluator of rewritten movie review sentences."},
        {"role": "user", "content": judge_prompt(original, candidate, target)},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        logits = llm(**inputs).logits[0, -1]
    probs = torch.softmax(logits[DIGIT_IDS].float(), dim=0)
    return float((probs * torch.arange(1, 6, device=probs.device)).sum())

def add_quality_metrics(df):
    df = df.copy()
    df["candidate"] = df["candidate"].fillna("").astype(str)
    df["target_confidence"] = batch_target_conf(df["candidate"].tolist(), df["target_label"].tolist())

    emb_o = embedder.encode(df["original"].tolist(), convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    emb_c = embedder.encode(df["candidate"].tolist(), convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    df["semantic_sim"] = (emb_o * emb_c).sum(dim=1).cpu().numpy()

    df["ppl_original"] = df["original"].map(perplexity)
    df["ppl_candidate"] = df["candidate"].map(perplexity)
    df["ppl_ratio"] = df["ppl_candidate"] / df["ppl_original"]

    n_o = df["original"].map(lambda t: len(words(t)))
    n_c = df["candidate"].map(lambda t: len(words(t)))
    df["length_ratio"] = n_c / n_o.clip(lower=1)

    df["judge_score"] = [judge_score(o, c, t) for o, c, t in
                         tqdm(zip(df["original"], df["candidate"], df["target_label"]), total=len(df), leave=False)]
    return df

DEFAULT_TH = {"sim": SIM_MIN, "rel_edit": REL_EDIT_MAX, "ppl_ratio": PPL_RATIO_MAX,
              "judge": JUDGE_MIN, "len_min": LEN_RATIO_MIN, "len_max": LEN_RATIO_MAX}

def check_columns(df, th=DEFAULT_TH):
    c = pd.DataFrame(index=df.index)
    c["valid"] = df["valid"].astype(bool)
    c["not_copy"] = df["edit_distance"] > 0
    c["similar"] = df["semantic_sim"] >= th["sim"]
    c["minimal"] = df["rel_edit_distance"] <= th["rel_edit"]
    c["fluent"] = df["ppl_ratio"] <= th["ppl_ratio"]
    c["length_ok"] = df["length_ratio"].between(th["len_min"], th["len_max"])
    c["judged_ok"] = df["judge_score"] >= th["judge"]
    return c

def failed_checks(row):
    return ", ".join(k for k, v in row.items() if not v)

In [ ]:
results_q = {}
for name, df in results.items():
    q = add_quality_metrics(df)
    checks = check_columns(q)
    q["actionable"] = checks.all(axis=1)
    q["failed_checks"] = checks.apply(failed_checks, axis=1)
    results_q[name] = q

pd.concat([q.assign(method=name) for name, q in results_q.items()]).to_csv("all_methods_quality.csv", index=False)

ok_all = pd.concat([q[q["valid"]] for q in results_q.values()])
ok_all[["edit_distance", "rel_edit_distance", "semantic_sim", "ppl_ratio", "length_ratio", "judge_score", "target_confidence"]].describe().round(2)

## Extended comparison table
- **Validity (%) and 95% CI**: Wilson interval, so the uncertainty on 250 examples is visible
- **Robust validity (%)**: valid and the classifier is at least `ROBUST_CONF` confident in the target label
- **Median edit, semantic sim, PPL ratio, judge score**: measured on valid counterfactuals
- **Actionable validity (%)**: share of all examples with a valid and actionable counterfactual
- **Actionable | valid (%)**: share of valid counterfactuals that are also actionable
- **Calls / tokens per valid or actionable CF**: total cost divided by the number of successes, a fairer cost than cost per example

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (c - h), 100 * (c + h)

def extended_summary(df, name):
    ok = df[df["valid"]]
    n = len(df)
    n_valid = int(df["valid"].sum())
    n_act = int(df["actionable"].sum())
    n_robust = int((df["valid"] & (df["target_confidence"] >= ROBUST_CONF)).sum())
    lo, hi = wilson(n_valid, n)
    total_tokens = (df["prompt_tokens"] + df["new_tokens"]).sum()
    return pd.DataFrame([{
        "Method": name,
        "Validity (%)": 100 * n_valid / n,
        "95% CI": f"[{lo:.1f}, {hi:.1f}]",
        "Robust validity (%)": 100 * n_robust / n,
        "Median edit (valid)": ok["edit_distance"].median(),
        "Semantic sim (valid)": ok["semantic_sim"].mean(),
        "PPL ratio (valid, median)": ok["ppl_ratio"].median(),
        "Judge score (valid)": ok["judge_score"].mean(),
        "Actionable validity (%)": 100 * n_act / n,
        "Actionable | valid (%)": 100 * n_act / n_valid if n_valid else np.nan,
        "Calls / valid CF": df["llm_calls"].sum() / n_valid if n_valid else np.nan,
        "Tokens / valid CF": total_tokens / n_valid if n_valid else np.nan,
        "Tokens / actionable CF": total_tokens / n_act if n_act else np.nan,
        "Mean flip confidence (valid)": ok["target_confidence"].mean(),
    }])

ext = pd.concat([extended_summary(q, name) for name, q in results_q.items()], ignore_index=True)
ext = ext.round(2)
ext.to_csv("summary_extended.csv", index=False)
ext.set_index("Method").T

## Are the differences real? Paired significance test
All methods run on the same examples, so we compare them with an exact McNemar test (a binomial test on the examples where exactly one of the two methods succeeds). `A only` = A succeeded and B did not. A small p-value means the difference is unlikely to be chance.

In [ ]:
from scipy.stats import binomtest
from itertools import combinations

def paired_test(a, b, col):
    m = a[["id", col]].merge(b[["id", col]], on="id", suffixes=("_a", "_b"))
    only_a = int((m[col + "_a"] & ~m[col + "_b"]).sum())
    only_b = int((~m[col + "_a"] & m[col + "_b"]).sum())
    n = only_a + only_b
    p = 1.0 if n == 0 else binomtest(only_a, n, 0.5).pvalue
    return only_a, only_b, p

rows = []
for (na, a), (nb, b) in combinations(results_q.items(), 2):
    for col, label in [("valid", "Validity"), ("actionable", "Actionable validity")]:
        oa, ob, p = paired_test(a, b, col)
        rows.append({"Metric": label, "A": na, "B": nb, "A only": oa, "B only": ob, "p-value": round(p, 4)})

pd.DataFrame(rows)

## Plot 4: meaning, fluency and plausibility of the valid counterfactuals
- **Left**: semantic similarity to the original (higher is better)
- **Middle**: perplexity of the counterfactual relative to the original (lower is better, 1 means as fluent as the original)
- **Right**: LLM-judge plausibility rating from 1 to 5 (higher is better)

In [ ]:
method_labels = ["One-shot", "Multi-\ncandidate", "Iterative\n(prev. output)", "Iterative\n(full history)"]
method_colors = ["orange", "tab:blue", "tab:red", "tab:green"]

panels = [("Semantic sim (valid)", "Semantic similarity to original", "Cosine similarity", 0.01),
          ("PPL ratio (valid, median)", "Fluency: perplexity ratio (median)", "Candidate PPL / original PPL", 0.02),
          ("Judge score (valid)", "LLM-judge plausibility", "Rating (1 to 5)", 0.03)]

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
for a, (col, title, ylabel, pad) in zip(ax, panels):
    vals = ext[col].to_numpy()
    a.bar(method_labels, vals, color=method_colors)
    a.set_title(title)
    a.set_ylabel(ylabel)
    a.set_ylim(0, vals.max() * 1.18)
    for i, v in enumerate(vals):
        a.text(i, v + vals.max() * pad, f"{v:.2f}", ha="center")

plt.tight_layout()
plt.savefig("quality_metrics.png", dpi=200, bbox_inches="tight")
plt.show()

## Plot 5 and the actionability funnel
Each column adds one more requirement on top of the previous one, so the last column is the actionable rate. The drop between two neighbouring columns shows which requirement removes the most valid counterfactuals.

In [ ]:
funnel_steps = ["valid", "not_copy", "similar", "minimal", "fluent", "length_ok", "judged_ok"]
funnel_names = ["Valid", "+ not a copy", "+ similar meaning", "+ minimal edit", "+ fluent", "+ length ok", "+ judge ok (actionable)"]

def funnel(df, th=DEFAULT_TH):
    c = check_columns(df, th)
    running = np.ones(len(df), dtype=bool)
    out = []
    for s in funnel_steps:
        running = running & c[s].to_numpy()
        out.append(100 * running.mean())
    return out

funnel_df = pd.DataFrame({name: funnel(q) for name, q in results_q.items()}, index=funnel_names).T.round(1)
display(funnel_df)

x = np.arange(len(results_q))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w / 2, ext["Validity (%)"], w, label="Valid", color="steelblue")
b2 = ax.bar(x + w / 2, ext["Actionable validity (%)"], w, label="Valid and actionable", color="seagreen")
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{bar.get_height():.0f}%", ha="center")
ax.set_xticks(x)
ax.set_xticklabels(method_labels)
ax.set_ylabel("Share of examples (%)")
ax.set_ylim(0, 110)
ax.set_title("Validity vs. actionable validity")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig("validity_vs_actionable.png", dpi=200, bbox_inches="tight")
plt.show()

## Sensitivity to the actionability thresholds
The thresholds are choices, so we recompute the actionable rate under a lenient, the default and a strict setting. If the ranking of the methods stays the same, the conclusion does not depend on the exact thresholds.

In [ ]:
presets = {
    "Lenient": {"sim": 0.5, "rel_edit": 0.7, "ppl_ratio": 5.0, "judge": 3.0, "len_min": 0.4, "len_max": 2.0},
    "Default": DEFAULT_TH,
    "Strict": {"sim": 0.75, "rel_edit": 0.4, "ppl_ratio": 2.0, "judge": 4.0, "len_min": 0.6, "len_max": 1.4},
}

sens = {}
for name, q in results_q.items():
    sens[name] = {p: 100 * check_columns(q, th).all(axis=1).mean() for p, th in presets.items()}

pd.DataFrame(sens).T.round(1).rename_axis("Actionable validity (%)")

## Looking at the examples
Valid counterfactuals that fail the actionability checks (with the reason), and valid counterfactuals that pass. Reading some of these is the best sanity check for the thresholds.

In [ ]:
pd.set_option("display.max_colwidth", 90)
show_cols = ["original", "candidate", "semantic_sim", "ppl_ratio", "judge_score", "failed_checks"]

q_iter = results_q["Iterative (previous output)"]

print("Valid but NOT actionable (iterative, previous output):")
bad = q_iter[q_iter["valid"] & ~q_iter["actionable"]]
display(bad[show_cols].sample(min(8, len(bad)), random_state=SEED).round(2))

print("Valid and actionable (iterative, previous output):")
good = q_iter[q_iter["actionable"]]
display(good[show_cols].sample(min(8, len(good)), random_state=SEED).round(2))

print("Most common reasons a valid counterfactual was not actionable (all methods):")
all_bad = pd.concat([q[q["valid"] & ~q["actionable"]] for q in results_q.values()])
all_bad["failed_checks"].str.split(", ").explode().value_counts()

# Answers to the research questions
The next cell prints the numbers behind each answer, so the written conclusions in the report can be taken directly from them.

In [ ]:
S = summary.set_index("Method")
multi = f"Multi-candidate (K={K})"

print("Q1: Does iterative repair improve validity over one-shot and multi-candidate selection?")
print(f"  One-shot:                 {S.loc['One-shot', 'Validity (%)']:.1f}%")
print(f"  Multi-candidate (K={K}):    {S.loc[multi, 'Validity (%)']:.1f}%")
print(f"  Iterative (prev. output): {S.loc['Iterative (previous output)', 'Validity (%)']:.1f}%")
print(f"  Iterative (full history): {S.loc['Iterative (full history)', 'Validity (%)']:.1f}%")

print("\nQ2: Does showing the full failure history help compared with only the previous output?")
print(f"  Validity: {S.loc['Iterative (previous output)', 'Validity (%)']:.1f}% -> {S.loc['Iterative (full history)', 'Validity (%)']:.1f}%")
print(f"  Mean attempts: {res_iter['attempts'].mean():.2f} -> {res_hist['attempts'].mean():.2f}")
print(f"  Tokens per example: {S.loc['Iterative (previous output)', 'Tokens / example']:.0f} -> {S.loc['Iterative (full history)', 'Tokens / example']:.0f}")

print("\nQ3: Trade-offs between validity, modification and cost")
for name in S.index:
    print(f"  {name:<30} validity {S.loc[name, 'Validity (%)']:5.1f}% | "
          f"edit distance {S.loc[name, 'Edit distance (valid)']:.2f} | "
          f"{S.loc[name, 'LLM calls / example']:.2f} calls | "
          f"{S.loc[name, 'Tokens / example']:.0f} tokens | "
          f"{S.loc[name, 'Seconds / example']:.2f} s")

## Q4: are the valid counterfactuals also actionable?

In [ ]:
E = ext.set_index("Method")

print("Q4: How many valid counterfactuals are also actionable, and what does that cost?")
for name in E.index:
    print(f"  {name:<30} valid {E.loc[name, 'Validity (%)']:5.1f}% | "
          f"actionable {E.loc[name, 'Actionable validity (%)']:5.1f}% | "
          f"actionable given valid {E.loc[name, 'Actionable | valid (%)']:5.1f}% | "
          f"semantic sim {E.loc[name, 'Semantic sim (valid)']:.2f} | "
          f"judge {E.loc[name, 'Judge score (valid)']:.2f} | "
          f"{E.loc[name, 'Tokens / actionable CF']:.0f} tokens per actionable CF")

# References
1. Dehghanighobadi, Z., Fischer, A., Zafar, M. B. (2025). *Can LLMs Explain Themselves Counterfactually?* EMNLP 2025, 7787–7815.
2. Qwen2.5-3B-Instruct – https://huggingface.co/Qwen/Qwen2.5-3B-Instruct
3. DistilBERT SST-2 classifier – https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
4. GLUE / SST-2 – https://huggingface.co/datasets/nyu-mll/glue
5. LangGraph – https://langchain-ai.github.io/langgraph/
6. Sentence-Transformers `all-MiniLM-L6-v2` – https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
7. Levenshtein distance – https://en.wikipedia.org/wiki/Levenshtein_distance
8. Wachter, S., Mittelstadt, B., Russell, C. (2017). *Counterfactual Explanations without Opening the Black Box.* Harvard Journal of Law & Technology.
9. Ustun, B., Spangher, A., Liu, Y. (2019). *Actionable Recourse in Linear Classification.* FAT* 2019.
10. Mothilal, R. K., Sharma, A., Tan, C. (2020). *Explaining Machine Learning Classifiers through Diverse Counterfactual Explanations.* FAT* 2020.
11. Karimi, A.-H., Barthe, G., Schölkopf, B., Valera, I. (2022). *A Survey of Algorithmic Recourse: Contrastive Explanations and Consequential Recommendations.* ACM Computing Surveys.
12. Radford, A. et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2).
13. Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* NeurIPS 2023.